# LSTM Data Preparation
## Atlanta / Birmingham Energy + Weather

This notebook is updated to match the **actual master CSV structure used in the EDA**.

Expected core columns include:

`timestamp`, `hourly_load_mw`, `date`, `hour`,
`temperature_c`, `dew_point_c`, `relative_humidity_percent`,
`station_pressure_kpa`, `wind_direction_deg`, `wind_speed_ms`,
`horizontal_visibility_km`, `hourly_precipitation_cm`

Additional weather variables are retained where useful.

### Important
Training is intentionally **not included**. GPU/CUDA setup and LSTM training will be handled separately.

The workflow is:

1. Load the EDA dataset
2. Validate the actual columns
3. Prepare timestamp and calendar features
4. Add time of day and season
5. Engineer weather features
6. Optionally create historical load features
7. Perform chronological 70/15/15 split
8. Scale using training data only
9. Create 24 hour LSTM sequences
10. Save arrays and scalers


In [29]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

# Change ONLY this section when switching cities.

CITY = "Atlanta"
FILE_PATH = "atlanta_master.csv"

# For Birmingham use:
# CITY = "Birmingham"
# FILE_PATH = "birmingham_final.csv"

TARGET = "hourly_load_mw"

SEQUENCE_LENGTH = 24

# Output files are automatically city-specific
CITY_PREFIX = CITY.lower().replace(" ", "_")

print("City:", CITY)
print("Input:", FILE_PATH)
print("Target:", TARGET)


City: Birmingham
Input: D:\Downloads\BTP\model_training\lstm_training_2\cleaning\birmingham_master.csv
Target: hourly_load_mw


In [31]:
# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(FILE_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()


Shape: (70080, 26)

Columns:
['timestamp', 'hourly_load_mw', 'date', 'hour', 'extraterrestrial_horizontal_radiation_wh_m2', 'extraterrestrial_direct_normal_radiation_wh_m2', 'global_horizontal_radiation_wh_m2', 'direct_normal_radiation_wh_m2', 'diffuse_horizontal_radiation_wh_m2', 'total_sky_cover_tenths', 'opaque_sky_cover_tenths', 'temperature_c', 'dew_point_c', 'relative_humidity_percent', 'station_pressure_kpa', 'wind_direction_deg', 'wind_speed_ms', 'horizontal_visibility_km', 'ceiling_height_m', 'observation_indicator', 'present_weather', 'precipitable_water_mm', 'broadband_aerosol_optical_depth', 'snow_depth_cm', 'days_since_last_snowfall', 'hourly_precipitation_cm']


,timestamp,hourly_load_mw,date,hour,extraterrestrial_horizontal_radiation_wh_m2,extraterrestrial_direct_normal_radiation_wh_m2,global_horizontal_radiation_wh_m2,direct_normal_radiation_wh_m2,diffuse_horizontal_radiation_wh_m2,total_sky_cover_tenths,opaque_sky_cover_tenths,temperature_c,dew_point_c,relative_humidity_percent,station_pressure_kpa,wind_direction_deg,wind_speed_ms,horizontal_visibility_km,ceiling_height_m,observation_indicator,present_weather,precipitable_water_mm,broadband_aerosol_optical_depth,snow_depth_cm,days_since_last_snowfall,hourly_precipitation_cm
0,2016-01-01 00:00:00,1221.158866,2016-01-01,1,0.0,0.0,NaN,NaN,NaN,10.0,10.0,7.2,4.4,83.0,100.1,360.0,2.3,9.7,2440.0,0,909999999,14.0,0.12,0.0,20.0,0.25
1,2016-01-01 01:00:00,1240.546048,2016-01-01,2,0.0,0.0,NaN,NaN,NaN,10.0,10.0,6.7,4.4,86.0,100.1,40.0,1.6,11.3,3050.0,0,999999999,14.0,0.12,0.0,20.0,0.00
2,2016-01-01 02:00:00,1202.009530,2016-01-01,3,0.0,0.0,NaN,NaN,NaN,10.0,10.0,7.2,3.9,80.0,100.1,110.0,2.8,11.3,1160.0,0,999999999,14.0,0.12,0.0,20.0,0.00
3,2016-01-01 03:00:00,1161.361693,2016-01-01,4,0.0,0.0,NaN,NaN,NaN,10.0,10.0,7.2,3.9,80.0,100.1,0.0,0.0,11.3,3050.0,0,999999999,14.0,0.12,0.0,20.0,0.00
4,2016-01-01 04:00:00,1116.248668,2016-01-01,5,0.0,0.0,NaN,NaN,NaN,10.0,10.0,6.7,3.9,83.0,100.1,0.0,0.0,11.3,3050.0,0,999999999,14.0,0.12,0.0,20.0,0.00


In [32]:
# ============================================================
# 4. VALIDATE REQUIRED COLUMNS
# ============================================================

required_columns = [
    "timestamp",
    "hourly_load_mw",
    "date",
    "hour",
    "temperature_c",
    "dew_point_c",
    "relative_humidity_percent",
    "station_pressure_kpa",
    "wind_direction_deg",
    "wind_speed_ms",
    "horizontal_visibility_km",
    "hourly_precipitation_cm"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        + "\n".join(missing_columns)
    )

print("All required columns are present.")


All required columns are present.


In [33]:
# ============================================================
# 5. DATETIME PREPARATION
# ============================================================

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df = df.sort_values("timestamp").reset_index(drop=True)

print("Start:", df["timestamp"].min())
print("End:", df["timestamp"].max())

print("\nMissing timestamps:", df["timestamp"].isna().sum())


Start: 2016-01-01 00:00:00
End: 2023-12-31 23:00:00

Missing timestamps: 0


In [34]:
# ============================================================
# 6. REMOVE DUPLICATES
# ============================================================

print("Duplicate rows before:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows after:", df.duplicated().sum())
print("Shape:", df.shape)


Duplicate rows before: 0
Duplicate rows after: 0
Shape: (70080, 26)


In [35]:
# ============================================================
# 7. CREATE CONSISTENT CALENDAR FEATURES
# ============================================================

# Use timestamp as the authoritative source.
# This avoids relying on the existing 1-24 hour column.

df["hour_lstm"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["day_of_month"] = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month
df["day_of_year"] = df["timestamp"].dt.dayofyear
df["week_of_year"] = df["timestamp"].dt.isocalendar().week.astype(int)

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

print(
    df[
        [
            "timestamp",
            "hour",
            "hour_lstm",
            "day_of_week",
            "month",
            "is_weekend"
        ]
    ].head(10)
)


            timestamp  hour  hour_lstm  day_of_week  month  is_weekend
0 2016-01-01 00:00:00     1          0            4      1           0
1 2016-01-01 01:00:00     2          1            4      1           0
2 2016-01-01 02:00:00     3          2            4      1           0
3 2016-01-01 03:00:00     4          3            4      1           0
4 2016-01-01 04:00:00     5          4            4      1           0
5 2016-01-01 05:00:00     6          5            4      1           0
6 2016-01-01 06:00:00     7          6            4      1           0
7 2016-01-01 07:00:00     8          7            4      1           0
8 2016-01-01 08:00:00     9          8            4      1           0
9 2016-01-01 09:00:00    10          9            4      1           0


In [36]:
# ============================================================
# 8. TIME OF DAY
# ============================================================

def get_time_period(hour):
    if 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    elif 18 <= hour < 22:
        return "Evening"
    else:
        return "Night"


df["time_period"] = df["hour_lstm"].apply(get_time_period)

df = pd.get_dummies(
    df,
    columns=["time_period"],
    dtype=int
)

print(
    df[
        [
            col for col in [
                "time_period_Morning",
                "time_period_Afternoon",
                "time_period_Evening",
                "time_period_Night"
            ]
            if col in df.columns
        ]
    ].head()
)


   time_period_Morning  time_period_Afternoon  time_period_Evening  time_period_Night
0                    0                      0                    0                  1
1                    0                      0                    0                  1
2                    0                      0                    0                  1
3                    0                      0                    0                  1
4                    0                      0                    0                  1


In [37]:
# ============================================================
# 9. SEASON
# ============================================================

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


df["season"] = df["month"].apply(get_season)

df = pd.get_dummies(
    df,
    columns=["season"],
    dtype=int
)

print("Season features created.")


Season features created.


In [38]:
# ============================================================
# 10. WEATHER FEATURE ENGINEERING
# ============================================================

# Temperature-dew point spread
df["temperature_dewpoint_diff"] = (
    df["temperature_c"] - df["dew_point_c"]
)

# Nonlinear temperature effect
df["temperature_squared"] = (
    df["temperature_c"] ** 2
)

# Nonlinear humidity effect
df["humidity_squared"] = (
    df["relative_humidity_percent"] ** 2
)

# Optional apparent moisture/temperature interaction
df["temperature_humidity_interaction"] = (
    df["temperature_c"]
    * df["relative_humidity_percent"]
)

print(
    df[
        [
            "temperature_c",
            "dew_point_c",
            "temperature_dewpoint_diff",
            "temperature_squared",
            "relative_humidity_percent",
            "humidity_squared",
            "temperature_humidity_interaction"
        ]
    ].head()
)


   temperature_c  dew_point_c  temperature_dewpoint_diff  temperature_squared  relative_humidity_percent  humidity_squared  temperature_humidity_interaction
0            7.2          4.4                        2.8                51.84                       83.0            6889.0                             597.6
1            6.7          4.4                        2.3                44.89                       86.0            7396.0                             576.2
2            7.2          3.9                        3.3                51.84                       80.0            6400.0                             576.0
3            7.2          3.9                        3.3                51.84                       80.0            6400.0                             576.0
4            6.7          3.9                        2.8                44.89                       83.0            6889.0                             556.1


In [39]:
# ============================================================
# 11. OPTIONAL HISTORICAL LOAD FEATURES
# ============================================================

# These are kept available for a comparison model.
# They should NOT automatically be used in the primary
# weather -> expected energy model.

df["load_lag_1"] = df[TARGET].shift(1)
df["load_lag_2"] = df[TARGET].shift(2)
df["load_lag_3"] = df[TARGET].shift(3)
df["load_lag_6"] = df[TARGET].shift(6)
df["load_lag_12"] = df[TARGET].shift(12)
df["load_lag_24"] = df[TARGET].shift(24)

# shift(1) prevents current target leakage
df["load_rolling_mean_3"] = (
    df[TARGET].shift(1).rolling(3).mean()
)

df["load_rolling_mean_6"] = (
    df[TARGET].shift(1).rolling(6).mean()
)

df["load_rolling_mean_12"] = (
    df[TARGET].shift(1).rolling(12).mean()
)

df["load_rolling_mean_24"] = (
    df[TARGET].shift(1).rolling(24).mean()
)


In [40]:
# ============================================================
# 12. FEATURE GROUPS
# ============================================================

# These are the core weather variables established during EDA.
weather_features = [
    "temperature_c",
    "dew_point_c",
    "relative_humidity_percent",
    "station_pressure_kpa",
    "wind_direction_deg",
    "wind_speed_ms",
    "horizontal_visibility_km",
    "hourly_precipitation_cm"
]

# Additional weather variables available in the master dataset.
# Include only if they survived the EDA / VIF feature-selection stage.
additional_weather_features = [
    "global_horizontal_radiation_wh_m2",
    "direct_normal_radiation_wh_m2",
    "diffuse_horizontal_radiation_wh_m2",
    "total_sky_cover_tenths",
    "opaque_sky_cover_tenths",
    "ceiling_height_m",
    "precipitable_water_mm",
    "broadband_aerosol_optical_depth",
    "snow_depth_cm",
    "days_since_last_snowfall"
]

calendar_features = [
    "hour_lstm",
    "day_of_week",
    "month",
    "day_of_year",
    "is_weekend"
]

time_features = [
    "time_period_Morning",
    "time_period_Afternoon",
    "time_period_Evening",
    "time_period_Night"
]

season_features = [
    "season_Winter",
    "season_Spring",
    "season_Summer",
    "season_Autumn"
]

engineered_weather_features = [
    "temperature_dewpoint_diff",
    "temperature_squared",
    "humidity_squared",
    "temperature_humidity_interaction"
]

historical_load_features = [
    "load_lag_1",
    "load_lag_2",
    "load_lag_3",
    "load_lag_6",
    "load_lag_12",
    "load_lag_24",
    "load_rolling_mean_3",
    "load_rolling_mean_6",
    "load_rolling_mean_12",
    "load_rolling_mean_24"
]


In [41]:
# ============================================================
# 13. PRIMARY FEATURE SET
# ============================================================

# PRIMARY MODEL:
# Weather + calendar + time-of-day + season + engineered weather
#
# This matches the project's current formulation:
# Given weather conditions, estimate how much energy SHOULD
# be consumed.

feature_columns = (
    weather_features
    + calendar_features
    + time_features
    + season_features
    + engineered_weather_features
)

feature_columns = [
    col for col in feature_columns
    if col in df.columns
]

print("PRIMARY FEATURE SET")
print("===================")
for i, col in enumerate(feature_columns, 1):
    print(f"{i:02d}. {col}")

print("\nNumber of primary features:", len(feature_columns))


PRIMARY FEATURE SET
01. temperature_c
02. dew_point_c
03. relative_humidity_percent
04. station_pressure_kpa
05. wind_direction_deg
06. wind_speed_ms
07. horizontal_visibility_km
08. hourly_precipitation_cm
09. hour_lstm
10. day_of_week
11. month
12. day_of_year
13. is_weekend
14. time_period_Morning
15. time_period_Afternoon
16. time_period_Evening
17. time_period_Night
18. season_Winter
19. season_Spring
20. season_Summer
21. season_Autumn
22. temperature_dewpoint_diff
23. temperature_squared
24. humidity_squared
25. temperature_humidity_interaction

Number of primary features: 25


In [42]:
# ============================================================
# 14. OPTIONAL: ADDITIONAL WEATHER FEATURES
# ============================================================

# Review these against the EDA/VIF results before enabling.
#
# If your EDA concluded that a variable should be excluded due
# to high VIF, redundancy, missingness, or low relevance, leave
# it out.

USE_ADDITIONAL_WEATHER = False

if USE_ADDITIONAL_WEATHER:
    additional_present = [
        col for col in additional_weather_features
        if col in df.columns
    ]

    feature_columns += additional_present

    print("Additional weather features added:")
    print(additional_present)


In [43]:
# ============================================================
# 15. OPTIONAL COMPARISON MODEL WITH HISTORICAL LOAD
# ============================================================

# This is NOT the recommended primary model.
# Enable only when explicitly comparing against the
# weather-only formulation.

USE_HISTORICAL_LOAD = False

if USE_HISTORICAL_LOAD:
    load_present = [
        col for col in historical_load_features
        if col in df.columns
    ]

    feature_columns += load_present

    print("Historical load features added:")
    print(load_present)

feature_columns = list(dict.fromkeys(feature_columns))

print("\nFinal feature count:", len(feature_columns))



Final feature count: 25


In [44]:
# ============================================================
# 16. REMOVE ROWS CREATED BY LAGS
# ============================================================

# If historical load features are disabled, the dataset itself
# does not need to lose 24 rows just because optional lag
# columns exist.

if USE_HISTORICAL_LOAD:
    df = df.dropna(
        subset=historical_load_features
    ).reset_index(drop=True)

# Always remove rows where the actual selected features or
# target are missing.
df = df.dropna(
    subset=feature_columns + [TARGET]
).reset_index(drop=True)

print("Shape after final feature cleaning:", df.shape)


Shape after final feature cleaning: (70080, 55)


In [45]:
# ============================================================
# 17. FINAL X AND y
# ============================================================

X = df[feature_columns].copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nMissing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())


X shape: (70080, 25)
y shape: (70080,)

Missing values in X: 0
Missing values in y: 0


In [46]:
# ============================================================
# 18. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# 70% train
# 15% validation
# 15% test
#
# NO RANDOM SHUFFLING.
# The chronological structure must be preserved.

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nDate ranges:")
print("Train:", df["timestamp"].iloc[0], "to", df["timestamp"].iloc[train_end - 1])
print("Val:  ", df["timestamp"].iloc[train_end], "to", df["timestamp"].iloc[val_end - 1])
print("Test: ", df["timestamp"].iloc[val_end], "to", df["timestamp"].iloc[-1])


Train: (49056, 25) (49056,)
Validation: (10512, 25) (10512,)
Test: (10512, 25) (10512,)

Date ranges:
Train: 2016-01-01 00:00:00 to 2021-08-07 23:00:00
Val:   2021-08-08 00:00:00 to 2022-10-19 23:00:00
Test:  2022-10-20 00:00:00 to 2023-12-31 23:00:00


In [47]:
# ============================================================
# 19. SCALE INPUT FEATURES
# ============================================================

X_scaler = MinMaxScaler()

# Fit ONLY on training data
X_train_scaled = X_scaler.fit_transform(X_train)

# Validation and test use the training scaler
X_val_scaled = X_scaler.transform(X_val)
X_test_scaled = X_scaler.transform(X_test)


In [48]:
# ============================================================
# 20. SCALE TARGET
# ============================================================

y_scaler = MinMaxScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = y_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.values.reshape(-1, 1)
)


In [49]:
# ============================================================
# 21. SAVE SCALERS
# ============================================================

joblib.dump(
    X_scaler,
    f"{CITY_PREFIX}_X_scaler.pkl"
)

joblib.dump(
    y_scaler,
    f"{CITY_PREFIX}_y_scaler.pkl"
)

print("Scalers saved:")
print(f"{CITY_PREFIX}_X_scaler.pkl")
print(f"{CITY_PREFIX}_y_scaler.pkl")


Scalers saved:
birmingham_X_scaler.pkl
birmingham_y_scaler.pkl


In [50]:
# ============================================================
# 22. CREATE LSTM SEQUENCES
# ============================================================

def create_sequences(X, y, sequence_length):
    X_sequences = []
    y_sequences = []

    for i in range(sequence_length, len(X)):
        X_sequences.append(
            X[i - sequence_length:i]
        )

        y_sequences.append(
            y[i]
        )

    return (
        np.asarray(X_sequences, dtype=np.float32),
        np.asarray(y_sequences, dtype=np.float32)
    )


In [51]:
# ============================================================
# 23. GENERATE LSTM SEQUENCES
# ============================================================

X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    SEQUENCE_LENGTH
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    SEQUENCE_LENGTH
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    SEQUENCE_LENGTH
)


In [52]:
# ============================================================
# 24. CHECK FINAL LSTM SHAPES
# ============================================================

print("X_train:", X_train_seq.shape)
print("y_train:", y_train_seq.shape)

print("X_val:", X_val_seq.shape)
print("y_val:", y_val_seq.shape)

print("X_test:", X_test_seq.shape)
print("y_test:", y_test_seq.shape)

print("\nExpected X format:")
print("(samples, 24, number_of_features)")


X_train: (49032, 24, 25)
y_train: (49032, 1)
X_val: (10488, 24, 25)
y_val: (10488, 1)
X_test: (10488, 24, 25)
y_test: (10488, 1)

Expected X format:
(samples, 24, number_of_features)


In [53]:
# ============================================================
# 25. SAVE PREPARED ARRAYS
# ============================================================

np.save(
    f"{CITY_PREFIX}_X_train.npy",
    X_train_seq
)

np.save(
    f"{CITY_PREFIX}_y_train.npy",
    y_train_seq
)

np.save(
    f"{CITY_PREFIX}_X_val.npy",
    X_val_seq
)

np.save(
    f"{CITY_PREFIX}_y_val.npy",
    y_val_seq
)

np.save(
    f"{CITY_PREFIX}_X_test.npy",
    X_test_seq
)

np.save(
    f"{CITY_PREFIX}_y_test.npy",
    y_test_seq
)

print("Prepared LSTM arrays saved successfully.")


Prepared LSTM arrays saved successfully.


In [54]:
# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print("=" * 60)
print("LSTM DATA PREPARATION COMPLETE")
print("=" * 60)

print("City:", CITY)
print("Rows:", len(df))
print("Features:", len(feature_columns))
print("Sequence length:", SEQUENCE_LENGTH)

print("\nFeature list:")
for col in feature_columns:
    print(" -", col)

print("\nSaved files:")
print(f" - {CITY_PREFIX}_X_scaler.pkl")
print(f" - {CITY_PREFIX}_y_scaler.pkl")
print(f" - {CITY_PREFIX}_X_train.npy")
print(f" - {CITY_PREFIX}_y_train.npy")
print(f" - {CITY_PREFIX}_X_val.npy")
print(f" - {CITY_PREFIX}_y_val.npy")
print(f" - {CITY_PREFIX}_X_test.npy")
print(f" - {CITY_PREFIX}_y_test.npy")


LSTM DATA PREPARATION COMPLETE
City: Birmingham
Rows: 70080
Features: 25
Sequence length: 24

Feature list:
 - temperature_c
 - dew_point_c
 - relative_humidity_percent
 - station_pressure_kpa
 - wind_direction_deg
 - wind_speed_ms
 - horizontal_visibility_km
 - hourly_precipitation_cm
 - hour_lstm
 - day_of_week
 - month
 - day_of_year
 - is_weekend
 - time_period_Morning
 - time_period_Afternoon
 - time_period_Evening
 - time_period_Night
 - season_Winter
 - season_Spring
 - season_Summer
 - season_Autumn
 - temperature_dewpoint_diff
 - temperature_squared
 - humidity_squared
 - temperature_humidity_interaction

Saved files:
 - birmingham_X_scaler.pkl
 - birmingham_y_scaler.pkl
 - birmingham_X_train.npy
 - birmingham_y_train.npy
 - birmingham_X_val.npy
 - birmingham_y_val.npy
 - birmingham_X_test.npy
 - birmingham_y_test.npy


## Switching from Atlanta to Birmingham

Only change the configuration cell:

```python
CITY = "Birmingham"
FILE_PATH = "birmingham_final.csv"
```

The output files will automatically receive the `birmingham_` prefix, so Atlanta files will not be overwritten.

## Important modelling decision

The default model uses:

**Weather + calendar + time-of-day + season + engineered weather → expected hourly load**

Historical load lags are disabled by default because the current project formulation is about estimating **expected consumption from weather conditions**, rather than simply forecasting load from its own history.

Before training, the final feature list should be checked against the EDA correlation and VIF results.
